# SENTINEL on Colab
Use a **T4 GPU** for 4-bit loading. This notebook treats Qwen output as evidence only after an undefended control reaches the attack and a benign control completes.

In [ ]:
!nvidia-smi -L

## 1 · Install
The clone command needs a public repository. If the repository is private, upload a source archive through Colab instead; do not paste a GitHub token into this notebook.

In [ ]:
!pip -q install uv
!git clone -q https://github.com/rayen-mansouri/Indaba.git
%cd Indaba
!uv sync --python 3.12 --extra hf
!uv pip install --python .venv/bin/python -q bitsandbytes accelerate

## 2 · Download Qwen3-8B weights
The adapter is offline-only (`local_files_only=True`), so cache the weights before running SENTINEL.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
!.venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('$MODEL_ID')"

## 3 · Deterministic sanity check

In [ ]:
MOCK_SCENARIO = "scenarios/public/finance/finance_false_approval.yaml"
!.venv/bin/sentinel run --scenario {MOCK_SCENARIO} --defense sentinel

## 4 · Real Qwen3-8B controls
`SENTINEL_HF_QUANT=4bit` uses NF4 so the model fits a 16 GB T4. Declare quantization in the report. The model is loaded once per evaluation process.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import files

os.environ["SENTINEL_HF_QUANT"] = "4bit"
OUT = Path("out")
ARTIFACTS = OUT / "artifacts"
OUT.mkdir(exist_ok=True)


def run_json(name, *arguments):
    command = [".venv/bin/sentinel", *arguments, "--artifacts", str(ARTIFACTS), "--json"]
    result = subprocess.run(command, check=True, capture_output=True, text=True)  # noqa: S603
    data = json.loads(result.stdout)
    (OUT / f"{name}.json").write_text(json.dumps(data, indent=2) + "\n")
    outcome = data["outcome"]
    print(name, "task_success=", outcome["task_success"], "attack_success=", outcome["attack_success"])
    return outcome

In [ ]:
ATTACK_SCENARIO = "scenarios/public/enterprise/enterprise_direct_token_request.yaml"
control = run_json(
    "attack-control",
    "run",
    "--scenario",
    ATTACK_SCENARIO,
    "--defense",
    "allow_all",
    "--model",
    "qwen3-8b",
)
if not control["attack_success"]:
    raise RuntimeError("STOP: undefended Qwen did not reach the attack; protected result would be vacuous")

In [ ]:
protected = run_json(
    "attack-protected",
    "run",
    "--scenario",
    ATTACK_SCENARIO,
    "--defense",
    "sentinel",
    "--model",
    "qwen3-8b",
)
if protected["attack_success"]:
    raise RuntimeError("SENTINEL did not prevent the attack in the protected run")

In [ ]:
BENIGN_SCENARIO = "scenarios/public/finance/finance_dispute_note.yaml"
benign = run_json(
    "benign-control",
    "run",
    "--scenario",
    BENIGN_SCENARIO,
    "--defense",
    "sentinel",
    "--attacker",
    "none",
    "--attack-mode",
    "none",
    "--model",
    "qwen3-8b",
)
if not benign["task_success"]:
    raise RuntimeError("STOP: benign Qwen control failed; do not claim live-model utility")

## 5 · Optional aggregate Qwen diagnostics
This is slow. Each split runs `allow_all` first and stops if no attack succeeds. Only then does it run SENTINEL. Do not report an ASR from a split whose undefended control reached zero attacks. The deterministic mock evaluation remains the reproducible primary benchmark.

In [ ]:
RUN_OPTIONAL_AGGREGATE = False


def run_scorecard(split, defense):
    output = OUT / f"{split}-{defense}-qwen.json"
    command = [
        ".venv/bin/sentinel",
        "eval",
        split,
        "--defense",
        defense,
        "--model",
        "qwen3-8b",
        "--artifacts",
        str(ARTIFACTS),
        "--output",
        str(output),
        "--json",
    ]
    subprocess.run(command, check=True, capture_output=True, text=True)  # noqa: S603
    return json.loads(output.read_text())


if RUN_OPTIONAL_AGGREGATE:
    for split in ("public", "validation"):
        undefended = run_scorecard(split, "allow_all")
        reached = sum(item["attack_success"] for item in undefended["outcomes"])
        if reached == 0:
            raise RuntimeError(f"STOP: {split} undefended aggregate reached zero attacks")
        run_scorecard(split, "sentinel")

## 6 · Build the dashboard and download the evidence

In [ ]:
!.venv/bin/python scripts/build_dashboard.py --extra out --out out/sentinel-dashboard.html
shutil.make_archive("sentinel_colab_out", "zip", "out")
files.download("sentinel_colab_out.zip")